In [1]:
# --- load the saved XGBoost model + its feature list ---
# We use the model.json file (xgboost's portable format) plus the meta.json
# sidecar so we know the exact feature order the model was trained on.
# At inference time we MUST hand the model a feature matrix whose columns
# are in this exact order — silently shuffled columns would feed garbage
# into the trees.

import json
import logging
import pickle
import time
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb

from polycluster import (
    build_history_df_from_orderfilled,
    compute_market_user_features,
    get_market_by_slug,
    get_market_orderfilled_events,
    parse_orderfilled_events,
)

CACHE_DIR    = Path("cache")
EVENTS_DIR   = CACHE_DIR / "events"
FEATURES_DIR = CACHE_DIR / "features"
MODEL_DIR    = CACHE_DIR / "models"
EVENTS_DIR.mkdir(parents=True, exist_ok=True)
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

# Filesystem-safe slug (some slugs are >200 chars and would blow ecryptfs/etc).
# MD5 keeps the suffix stable across Python processes; the built-in hash() is
# randomized per PYTHONHASHSEED and produced different cache filenames per run.
def slug_key(slug: str) -> str:
    if len(slug) <= 180:
        return slug
    h = hashlib.md5(slug.encode()).hexdigest()[:8]
    return f"{slug[:170]}__{h}"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
log = logging.getLogger("tryboost")

# Load model + meta
model = xgb.XGBClassifier()
model.load_model(str(MODEL_DIR / "xgb_insider_latest.json"))

with open(MODEL_DIR / "xgb_insider_latest.meta.json") as f:
    meta = json.load(f)

FEATURE_COLS: list[str] = meta["features"]
THRESHOLD = 0.7

log.info(f"Loaded model trained at {meta['trained_at']}")
log.info(f"  n_features = {meta['n_features']}")
log.info(f"  ROC-AUC = {meta['metrics']['roc_auc']:.4f} · PR-AUC = {meta['metrics']['pr_auc']:.4f}")
log.info(f"  scale_pos_weight = {meta['scale_pos_weight']:.2f}")
log.info(f"Flagging threshold: pred_prob_insider >= {THRESHOLD}")

09:30:16  Loaded model trained at 20260522_170006
09:30:16    n_features = 319
09:30:16    ROC-AUC = 0.9492 · PR-AUC = 0.8399
09:30:16    scale_pos_weight = 5.74
09:30:16  Flagging threshold: pred_prob_insider >= 0.7


In [3]:
# --- fetch raw OrderFilled events for the target market ---
# Same cache shape as xgboost.ipynb so the two notebooks share cache/events/.
# - If cache/events/<slug>.pkl exists → just load it (cheap).
# - Otherwise → call get_market_orderfilled_events with verbose=True so the
#   new blue heartbeat surfaces while we wait.
# Replace MARKET_SLUG with the slug you actually want to scan.

MARKET_SLUG = "will-zelenskyy-wear-a-suit-before-july"

def fetch_events_cached(slug: str):
    """Returns (market, events). Reads from disk if cached, else fetches + writes."""
    cache_path = EVENTS_DIR / f"{slug_key(slug)}.pkl"
    if cache_path.exists():
        with open(cache_path, "rb") as f:
            return pickle.load(f)
    market = get_market_by_slug(slug)
    result = get_market_orderfilled_events(market, checkpointed=True, verbose=True)
    events = result["rows"] if isinstance(result, dict) else result
    with open(cache_path, "wb") as f:
        pickle.dump((market, events), f)
    return (market, events)

t0 = time.time()
cache_path = EVENTS_DIR / f"{slug_key(MARKET_SLUG)}.pkl"
was_cached = cache_path.exists()

market, events = fetch_events_cached(MARKET_SLUG)

log.info(
    f"{'CACHED ' if was_cached else 'FETCHED'}  "
    f"{len(events):,} events  ·  {time.time()-t0:.1f}s  ·  {MARKET_SLUG}"
)
log.info(
    f"market: closed={market.closed} · "
    f"window=[{market.start_ts}, {market.end_ts}] "
    f"({(market.end_ts - market.start_ts)/86400:.1f}d) · "
    f"{len(market.token_ids)} token(s)"
)

[polycluster:will-zelenskyy-wear-a-suit-before-july] get_orderfilled_events_checkpointed: 2 token(s), sides=['maker', 'taker'], window=[1747953080, 1752021039] (4067959s), chunk=1800s
[polycluster:will-zelenskyy-wear-a-suit-before-july] (1/4) token 1/2 side=maker: starting from 1747953080
[polycluster:will-zelenskyy-wear-a-suit-before-july]   token 1 side=maker chunk 1/2260 [1747953080, 1747954879]: 0 rows (running total 0) in 0.3s
[polycluster:will-zelenskyy-wear-a-suit-before-july]   token 1 side=maker chunk 2/2260 [1747954880, 1747956679]: 0 rows (running total 0) in 0.2s
[polycluster:will-zelenskyy-wear-a-suit-before-july]   token 1 side=maker chunk 3/2260 [1747956680, 1747958479]: 1 rows (running total 1) in 0.2s
[polycluster:will-zelenskyy-wear-a-suit-before-july]   token 1 side=maker chunk 4/2260 [1747958480, 1747960279]: 1 rows (running total 2) in 0.2s
[polycluster:will-zelenskyy-wear-a-suit-before-july]   token 1 side=maker chunk 5/2260 [1747960280, 1747962079]: 1 rows (runni

10:07:56  FETCHED  112,589 events  ·  2221.6s  ·  will-zelenskyy-wear-a-suit-before-july
10:07:56  market: closed=True · window=[1747953080, 1752021039] (47.1d) · 2 token(s)


In [4]:
# --- identify candidate wallets to score ---
# Strategy: take every wallet that ever appeared as maker or taker in this
# market, rank them by total token volume, and keep the top N. Tiny wallets
# don't produce meaningful features (most behavioral signals need >= a few
# trades), and they're rarely insiders anyway.

TOP_N = 200

parsed = parse_orderfilled_events(events, market.token_ids, market.outcomes)

# rank_by_total_amount is a list of (wallet, summary_dict), sorted desc by
# total token amount traded. We just want the wallets.
ranked = parsed["rank_by_total_amount"]
all_wallets = [w for w, _ in ranked]
candidate_wallets = [w.lower() for w in all_wallets[:TOP_N]]

log.info(f"{len(all_wallets):,} unique wallets traded this market")
log.info(f"Scoring the top {len(candidate_wallets)} by total volume")

# Show the top 10 by volume for sanity
print("\nTop 10 wallets by volume in this market:")
for i, (w, summ) in enumerate(ranked[:10], 1):
    print(f"  {i:2d}. {w}  ·  total_amount={summ['total_amount']:,.0f} tokens  ·  trades={int(summ['total_activity'])}")

11:10:56  8,793 unique wallets traded this market
11:10:56  Scoring the top 200 by total volume



Top 10 wallets by volume in this market:
   1. 0x4bfb41d5b3570defd03c39a9a4d8de6bd8b8982e  ·  total_amount=238,272,126 tokens  ·  trades=39403
   2. 0xe6e4680bac32fb22b6886e846654811a9f5c0ee5  ·  total_amount=52,893,387 tokens  ·  trades=2880
   3. 0x24c8cf69a0e0a17eee21f69d29752bfa32e823e1  ·  total_amount=12,450,769 tokens  ·  trades=2072
   4. 0x5bffcf561bcae83af680ad600cb99f1184d6ffbe  ·  total_amount=11,486,331 tokens  ·  trades=641
   5. 0xecb14ac6e9ca447ce2f2912e6217b43d7b655da3  ·  total_amount=9,954,439 tokens  ·  trades=881
   6. 0x2373809dadc2c73d05038df89e9399560f445b7f  ·  total_amount=9,934,418 tokens  ·  trades=562
   7. 0x011f2d377e56119fb09196dffb0948ae55711122  ·  total_amount=8,926,375 tokens  ·  trades=1552
   8. 0x889e7f0464c72eb8cda1525ebc12b6aaba9d09e0  ·  total_amount=8,906,616 tokens  ·  trades=718
   9. 0x75049bd489194be19c45c31ed311e556411c9c69  ·  total_amount=8,886,156 tokens  ·  trades=490
  10. 0xf0ec554fe75d57fef3f2404fce070b5c71d46064  ·  total_amount=

In [5]:
# --- compute the 319 model features for every candidate wallet ---
# We reuse the same cache/features/<slug>.parquet file as xgboost.ipynb. If
# the file already has rows for some of our candidate wallets (e.g. labeled
# training wallets), we keep them. We only do the expensive per-wallet
# feature computation for wallets that aren't already in the parquet, then
# union the new rows back in and rewrite the file.

def features_for_wallets(slug: str, wallets: list[str], market, events) -> pd.DataFrame:
    """Return a feature row per wallet. Extends cache/features/<slug>.parquet."""
    feat_path = FEATURES_DIR / f"{slug_key(slug)}.parquet"
    wallets = [w.lower() for w in wallets]
    wallets_set = set(wallets)

    if feat_path.exists():
        existing = pd.read_parquet(feat_path)
        existing_wallets = set(existing["wallet"].str.lower())
        missing = sorted(wallets_set - existing_wallets)
        log.info(f"  cache has features for {len(existing_wallets - wallets_set | existing_wallets & wallets_set):,} wallet(s)  ·  need to compute {len(missing):,} new")
    else:
        existing = None
        missing = sorted(wallets_set)
        log.info(f"  no feature cache yet  ·  computing {len(missing):,} wallet(s) from scratch")

    if missing:
        # Auto-derive final_outcome_yes for closed markets
        final_outcome_yes = None
        if market.closed and market.outcome_prices:
            final_outcome_yes = 1 if float(market.outcome_prices[0]) >= 0.5 else 0

        history_df = build_history_df_from_orderfilled(
            events,
            yes_token=market.yes_token_id,
            no_token=market.no_token_id,
        )

        new_rows = []
        n_empty = n_err = 0
        for w in missing:
            trades = parsed["wallet_trades"].get(w, [])
            if not trades:
                new_rows.append({"wallet": w, "market_slug": slug})
                n_empty += 1
                continue
            try:
                feats = compute_market_user_features(
                    parsed_trades=trades,
                    history_df=history_df,
                    market_start_time=market.start_ts,
                    market_end_time=market.end_ts,
                    final_outcome_yes=final_outcome_yes,
                )
                feats["wallet"] = w
                feats["market_slug"] = slug
                new_rows.append(feats)
            except Exception as e:
                n_err += 1
                log.warning(f"    feat fail wallet={w[:10]}.. {type(e).__name__}: {e}")
                new_rows.append({"wallet": w, "market_slug": slug})

        new_df = pd.DataFrame(new_rows)
        combined = pd.concat([existing, new_df], ignore_index=True) if existing is not None else new_df
        combined.to_parquet(feat_path, index=False)
        log.info(f"  computed {len(new_df):,} new rows ({n_empty} empty / {n_err} errored)  ·  cache now {len(combined):,} rows")
    else:
        combined = existing

    return combined[combined["wallet"].str.lower().isin(wallets_set)].reset_index(drop=True)

t0 = time.time()
log.info(f"=== Computing features for {len(candidate_wallets)} candidate wallets ===")
df_feats = features_for_wallets(MARKET_SLUG, candidate_wallets, market, events)
log.info(f"=== Done in {time.time()-t0:.1f}s · {len(df_feats)} feature rows ready ===")

11:10:56  === Computing features for 200 candidate wallets ===
11:10:56    no feature cache yet  ·  computing 200 wallet(s) from scratch


KeyboardInterrupt: 

In [ ]:
# --- score every candidate with the XGBoost model ---
# Three things to be careful about:
#   1. Column order MUST match meta["features"] exactly (the model has no
#      feature-name awareness once we hand it a numpy array — it's positional).
#   2. Missing feature columns (a feature that exists in meta but not in our
#      parquet, e.g. because no wallet had data to derive it) get filled with
#      NaN. XGBoost handles NaN natively — it learned a default direction at
#      each tree split.
#   3. We use predict_proba directly (not score_new_rows, which calls
#      decision_function and would error on XGBoost — XGB doesn't expose it).

# Ensure every required feature column is present, in the right order.
for col in FEATURE_COLS:
    if col not in df_feats.columns:
        df_feats[col] = np.nan

X_new = df_feats[FEATURE_COLS].astype(float)
proba = model.predict_proba(X_new)[:, 1]

scored = df_feats[["wallet", "market_slug"]].copy()
scored["pred_prob_insider"] = proba
scored = scored.sort_values("pred_prob_insider", ascending=False).reset_index(drop=True)

log.info(f"Scored {len(scored)} wallets")
log.info(f"  max score = {scored['pred_prob_insider'].max():.4f}")
log.info(f"  median    = {scored['pred_prob_insider'].median():.4f}")
log.info(f"  >= {THRESHOLD}: {(scored['pred_prob_insider'] >= THRESHOLD).sum()}")

In [ ]:
# ---show the flagged wallets and a top-N ranked table ---
# Two views:
#   FLAGGED: every wallet at or above the threshold — the actionable list.
#   TOP 20:  the ranked head regardless of threshold, so you can see how
#            close the next-best wallets are (sometimes the threshold cuts
#            through a dense cluster of suspicious wallets and you want
#            eyeballs on them anyway).

flagged = scored[scored["pred_prob_insider"] >= THRESHOLD].reset_index(drop=True)

print("=" * 76)
print(f"FLAGGED wallets (pred_prob_insider >= {THRESHOLD})  ·  market: {MARKET_SLUG}")
print("=" * 76)
if flagged.empty:
    print("(none)")
else:
    for i, row in flagged.iterrows():
        print(f"  {i+1:3d}. {row['wallet']}   p={row['pred_prob_insider']:.4f}")

print()
print("Top 20 ranked candidates (for inspection):")
display(scored.head(20).style.format({"pred_prob_insider": "{:.4f}"}))